In [2]:
import pandas as pd
import os
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch


from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [4]:
X = pd.read_csv('/Users/natalialogvinova/Desktop/DS diploma/X_train_tifidf_combined_2910.csv', sep = ',')

In [5]:
X

,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,323,324,325,326,327,328,329,330,331,332
0,0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.310143,0.083895,0.147548,0.033233,0.000000,0.0,0.00000,0.000000,0.160753
1,1,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.227276,0.000000,0.270311,0.121766,0.000000,0.0,0.00000,0.000000,0.000000
2,2,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.126605,0.000000,0.0,0.00000,0.000000,0.000000
3,3,1.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.278447,0.000000,0.000000,0.149182,0.000000,0.0,0.00000,0.000000,0.000000
4,4,1.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.094846,0.000000,0.000000,0.050815,0.000000,0.0,0.00000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265638,265638,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.128045,0.000000,0.0,0.00000,0.000000,0.000000
265639,265639,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.069677,0.000000,0.165741,0.037330,0.092795,0.0,0.09327,0.090164,0.000000
265640,265640,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.071310,0.073833,0.099861,0.175627,0.039557,0.000000,0.0,0.00000,0.000000,0.000000
265641,265641,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.200077,0.000000,0.000000,0.000000,0.110987,0.000000,0.0,0.00000,0.268066,0.000000


In [6]:
columns_to_drop = ['Unnamed: 0']
X.drop(columns=columns_to_drop, inplace=True)

In [7]:
y = pd.read_csv('/Users/natalialogvinova/Desktop/DS diploma/y_train_2910.csv', sep = ',')

In [8]:
y.drop('Unnamed: 0', axis = 1, inplace = True)

In [9]:
# Объединяем в один DataFrame
df_combined = X.copy()
df_combined['target'] = y.values 

# Создаем Dataset
dataset = TensorDataset(
    torch.FloatTensor(X.values),
    torch.LongTensor(y.values.flatten())  # или .FloatTensor() для регрессии
)

In [10]:
X_test = pd.read_csv('/Users/natalialogvinova/Desktop/DS diploma/X_test_tifidf_combined_2910.csv', sep = ',')

In [11]:
columns_to_drop = ['Unnamed: 0']
X_test.drop(columns=columns_to_drop, inplace=True)

In [12]:
y_test = pd.read_csv('/Users/natalialogvinova/Desktop/DS diploma/y_test_2910.csv', sep = ',')

In [13]:
y_test.drop('Unnamed: 0', axis = 1, inplace = True)

In [14]:
# Объединяем в один DataFrame
df_combined_val = X_test.copy()
df_combined_val['target'] = y_test.values 

# Создаем Dataset
dataset_val = TensorDataset(
    torch.FloatTensor(X_test.values),
    torch.LongTensor(y_test.values.flatten())  # или .FloatTensor() для регрессии
)

In [15]:
from torch.utils.data import DataLoader

batch_size_train = 64
batch_size_test = 1000

train_hh_loader = DataLoader(
    dataset=dataset,
    batch_size=batch_size_train,
    shuffle=True,
    drop_last=True
)

val_hh_loader = DataLoader(
    dataset=dataset_val,
    batch_size=batch_size_test,
    shuffle=False,
    drop_last=False
)

In [16]:
train_hh_loader

In [17]:
for batch in train_hh_loader:
    feats, labels = batch  # предполагая, что данные возвращают (features, targets)
    print("Форма батча:", feats.shape)
    print("Количество признаков:", feats.shape[1])  # второе измерение - это features
    break  # важно: break после первого батча!

Форма батча: torch.Size([64, 333])
Количество признаков: 333


In [18]:
feats, labels = next(iter(train_hh_loader))
feats.shape, labels.shape

(torch.Size([64, 333]), torch.Size([64]))

In [19]:
def train_epoch(
    network,
    train_loader,
    criterion,
    optimizer,
):
    """
        Функция обучения `network` с `optimizer` 1 эпоху с данными из `train_loader` для минимизации `criterion`
    """

    # переводим модель в режим обучения (может влиять на батч нормализацию, дропаут)
    network.train()
    total_loss = 0
    for feats, labels in train_loader:
        # 0. распакавываем данные на нужное устройство
        labels = labels.float()
        feats, labels = feats.to(device), labels.to(device)

        # 1. сбрасываем градиенты предыдущего батча
        optimizer.zero_grad()

        # 2. прогоняем данные через нашу нейросеть
        logits = network(feats)
        #logits = logits.squeeze()
        labels = labels.squeeze()

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

        mean_loss = total_loss / len(train_loader)
    print(f'Mean Train Loss: {mean_loss:.6f}')
    return total_loss / len(train_loader)

In [20]:
@torch.no_grad()
def val_epoch(
    network,
    val_loader,
    criterion,
    
):
    """
        Считаем лосс и accuracy на валидационной выборке
    """

    val_loss = 0
    all_predictions = []
    all_targets = []

    # переводим модель в режим теста
    network.eval()
    for feats, labels in val_loader:
        labels = labels.float()
        # 0. распакавываем данные на нужное устройство
        feats, labels = feats.to(device), labels.to(device)

        # 1. прогоняем данные через нашу нейросеть
        logits = network(feats)
        #logits = logits.squeeze()
        labels = labels.squeeze()

        # Сохраняем предсказания и истинные значения для метрик
        all_predictions.extend(logits.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

    # Вычисляем метрики
    all_predictions = np.array(all_predictions)
    all_targets = np.array(all_targets)
    
    # R² score
    r2 = r2_score(all_targets, all_predictions)
    
    # RMSE
    rmse = np.sqrt(np.mean((all_targets - all_predictions) ** 2))

    val_loss += criterion(logits, labels).item()
    
    print(f'Val Loss: {val_loss:.6f}, R²: {r2:.4f}, RMSE: {rmse:.4f}')
    return val_loss, r2, rmse

In [21]:
def train_val(
    network,
    n_epochs,
    criterion,
    optimizer,
    train_loader,
    val_loader,
):
    """
        Полный цикл обучения нейронной сети
    """

    #val_epoch(network, val_loader, criterion)
    for epoch in range(1, n_epochs + 1):
        train_epoch(network, train_loader, criterion, optimizer)
        val_epoch(network, val_loader, criterion)

In [22]:
import torch.nn as nn

class SimpleRegression(nn.Module):
    def __init__(self, input_size=333):
        super().__init__()
        # Всего 1 выходной нейрон для предсказания одного числа
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x):
        x = self.linear(x)

        return x

In [23]:
model = SimpleRegression().to(device)

In [24]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:
train_val(model, 4, criterion, optimizer, train_hh_loader, val_hh_loader)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Mean Train Loss: 6772808373.790843


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([411])) that is different to the input size (torch.Size([411, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Val Loss: 8748345344.000000, R²: -2.0680, RMSE: 82495.1641


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


...

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([411])) that is different to the input size (torch.Size([411, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Val Loss: 8730657792.000000, R²: -2.0604, RMSE: 82393.6719


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Mean Train Loss: 6751917531.481445
Val Loss: 8721826816.000000, R²: -2.0565, RMSE: 82345.8516


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([411])) that is different to the input size (torch.Size([411, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


## Добавим 2 слоя (334 -- 128 -- 1) через ReLu

In [26]:
class TwoLayerRegression(nn.Module):
    def __init__(self, input_size=333, hidden_size=128):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, 1)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)   

        return x

In [27]:
model_2 = TwoLayerRegression().to(device)

In [28]:
train_val(model_2, 10, criterion, optimizer, train_hh_loader, val_hh_loader)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Mean Train Loss: 6780140338.459759


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([411])) that is different to the input size (torch.Size([411, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Val Loss: 8757193728.000000, R²: -2.0719, RMSE: 82547.8828


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


...

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([411])) that is different to the input size (torch.Size([411, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Val Loss: 8757193728.000000, R²: -2.0719, RMSE: 82547.8828


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Mean Train Loss: 6780757501.285783
Val Loss: 8757193728.000000, R²: -2.0719, RMSE: 82547.8828


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([411])) that is different to the input size (torch.Size([411, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


## Добавим нормализацию

In [29]:
class TwoLayerRegressionNorm(nn.Module):
    def __init__(self, input_size=333, hidden_size=128):
        super().__init__()
        self.normalize = nn.BatchNorm1d(input_size)
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, 1)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.normalize(x)
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)   

        return x

In [30]:
model_3 = TwoLayerRegressionNorm().to(device)

In [31]:
train_val(model_3, 10, criterion, optimizer, train_hh_loader, val_hh_loader)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Mean Train Loss: 6779938890.702651


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([411])) that is different to the input size (torch.Size([411, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Val Loss: 8757189632.000000, R²: -2.0719, RMSE: 82547.8828


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


...

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([411])) that is different to the input size (torch.Size([411, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Val Loss: 8757189632.000000, R²: -2.0719, RMSE: 82547.8672


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Mean Train Loss: 6780728957.347470


KeyboardInterrupt: 

## Попробуем нормализовать данные на этапе загрузки в даталоадер

In [33]:
# ДИАГНОСТИКА ДАННЫХ С ПРАВИЛЬНЫМИ ТИПАМИ
print("=== ДИАГНОСТИКА ДАННЫХ ===")

all_features = []
all_targets = []

# Собираем данные
for feats, labels in train_hh_loader:
    all_features.append(feats)
    all_targets.append(labels)

# Конкатенируем и преобразуем в float
all_features = torch.cat(all_features).float()  # ← ДОБАВИТЬ .float()
all_targets = torch.cat(all_targets).float()    # ← ДОБАВИТЬ .float()

print(f"Features shape: {all_features.shape}")
print(f"Targets shape: {all_targets.shape}")

# Теперь mean() будет работать
print(f"\nFeatures statistics:")
print(f"  Min: {all_features.min():.2f}")
print(f"  Max: {all_features.max():.2f}") 
print(f"  Mean: {all_features.mean():.2f}")
print(f"  Std: {all_features.std():.2f}")

print(f"\nTargets statistics:")
print(f"  Min: {all_targets.min():.2f}")
print(f"  Max: {all_targets.max():.2f}")
print(f"  Mean: {all_targets.mean():.2f}")
print(f"  Std: {all_targets.std():.2f}")

# Проверим наличие NaN/Inf
print(f"\nData quality:")
print(f"  NaN in features: {torch.isnan(all_features).any()}")
print(f"  Inf in features: {torch.isinf(all_features).any()}")
print(f"  NaN in targets: {torch.isnan(all_targets).any()}")
print(f"  Inf in targets: {torch.isinf(all_targets).any()}")

=== ДИАГНОСТИКА ДАННЫХ ===
Features shape: torch.Size([265600, 333])
Targets shape: torch.Size([265600])

Features statistics:
  Min: 0.00
  Max: 6.00
  Mean: 0.04
  Std: 0.26

Targets statistics:
  Min: 1.00
  Max: 1500000.00
  Mean: 67727.55
  Std: 46838.99

Data quality:
  NaN in features: False
  Inf in features: False
  NaN in targets: False
  Inf in targets: False


In [34]:
class RobustDataFrameTransformer:
    def __init__(self, features_df, targets_series, clip_percentile=99):
        """
        features_df: DataFrame с фичами
        targets_series: Series с целевой переменной
        clip_percentile: процентиль для обрезки выбросов
        """
        # Конвертируем в numpy
        features_array = features_df.values.astype(np.float32)
        targets_array = targets_series.values.astype(np.float32)
        
        # ОБРЕЗАЕМ ВЫБРОСЫ перед вычислением статистик
        self.features_clip_value = np.percentile(np.abs(features_array), clip_percentile, axis=0)
        self.targets_clip_value = np.percentile(np.abs(targets_array), clip_percentile)
        
        features_clipped = np.clip(features_array, -self.features_clip_value, self.features_clip_value)
        targets_clipped = np.clip(targets_array, -self.targets_clip_value, self.targets_clip_value)
        
        # Вычисляем статистики на ОБРЕЗАННЫХ данных
        self.features_mean = np.mean(features_clipped, axis=0)
        self.features_std = np.std(features_clipped, axis=0)
        self.targets_mean = np.mean(targets_clipped)
        self.targets_std = np.std(targets_clipped)
        
        # Защита от деления на 0
        self.features_std = np.where(self.features_std < 1e-8, 1.0, self.features_std)
        if self.targets_std < 1e-8:
            self.targets_std = 1.0
            
        self.feature_columns = features_df.columns.tolist()
        self.clip_percentile = clip_percentile
        
        print(f"RobustDataFrameTransformer создан (clip {clip_percentile}%):")
        print(f"  Features: {len(self.feature_columns)} columns")
        print(f"  Targets: mean={self.targets_mean:.2f}, std={self.targets_std:.2f}")
        print(f"  Clip values - features: {np.max(self.features_clip_value):.2f}, targets: {self.targets_clip_value:.2f}")
    
    def transform_features(self, features_df):
        """Нормализует DataFrame с обрезкой выбросов"""
        features_np = features_df.values.astype(np.float32)
        # Сначала обрезаем выбросы
        features_clipped = np.clip(features_np, -self.features_clip_value, self.features_clip_value)
        # Потом нормализуем
        features_norm = (features_clipped - self.features_mean) / self.features_std
        return features_norm
    
    def transform_targets(self, targets_series):
        """Нормализует Series с обрезкой выбросов"""
        targets_np = targets_series.values.astype(np.float32)
        targets_clipped = np.clip(targets_np, -self.targets_clip_value, self.targets_clip_value)
        targets_norm = (targets_clipped - self.targets_mean) / self.targets_std
        return targets_norm
    
    def inverse_transform_targets(self, targets_norm):
        """Денормализует предсказания"""
        if isinstance(targets_norm, torch.Tensor):
            targets_norm = targets_norm.detach().cpu().numpy()
        return targets_norm * self.targets_std + self.targets_mean

In [35]:


print("=== ПОДГОТОВКА ДАННЫХ ИЗ DATAFRAME ===")

# Шаг 1: Проверяем данные
print("Исходные данные:")
print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")

# Шаг 2: Создаем трансформер на ВСЕХ данных
print("\n=== СОЗДАНИЕ ТРАНСФОРМЕРА ===")
transformer = RobustDataFrameTransformer(X, y)

# Шаг 3: Нормализуем данные
print("\n=== НОРМАЛИЗАЦИЯ ДАННЫХ ===")
normalized_features = transformer.transform_features(X)
normalized_targets = transformer.transform_targets(y)

print(f"Нормализованные features: {normalized_features.min():.3f} to {normalized_features.max():.3f}")
print(f"Нормализованные targets: {normalized_targets.min():.3f} to {normalized_targets.max():.3f}")

# Шаг 4: Преобразуем в тензоры и создаем DataLoader
print("\n=== СОЗДАНИЕ DATALOADER ===")
# Конвертируем в тензоры PyTorch
features_tensor = torch.FloatTensor(normalized_features)
targets_tensor = torch.FloatTensor(normalized_targets)

# Создаем Dataset
dataset = TensorDataset(features_tensor, targets_tensor)

# Разделяем на train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Создаем DataLoader
batch_size = 64
train_hh_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_hh_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print("✅ DataLoader созданы из DataFrame!")

=== ПОДГОТОВКА ДАННЫХ ИЗ DATAFRAME ===
Исходные данные:
Features shape: (265643, 333)
Targets shape: (265643, 1)

=== СОЗДАНИЕ ТРАНСФОРМЕРА ===
RobustDataFrameTransformer создан (clip 99%):
  Features: 333 columns
  Targets: mean=66998.06, std=41595.69
  Clip values - features: 6.00, targets: 250000.00

=== НОРМАЛИЗАЦИЯ ДАННЫХ ===
Нормализованные features: -4.088 to 9.868
Нормализованные targets: -1.611 to 4.400

=== СОЗДАНИЕ DATALOADER ===
Train samples: 212514
Val samples: 53129
✅ DataLoader созданы из DataFrame!


In [36]:
model_4 = TwoLayerRegressionNorm().to(device)

In [ ]:
train_val(model_4, 10, criterion, optimizer, train_hh_loader, val_hh_loader)